# m12i snapshot 600: direct vs fitted v_travel

Plots the CSVs written by `scc/04_direct_m12i.py` (direct) and `scc/03_fit_m12i.py` (fit). Nothing is recomputed here, so change the formatting freely and re-run.

**Direct v_travel** = −(mass-weighted mean Cartesian velocity of the selected star particles, host frame).
**Fit** = reflex vector from the 9-parameter likelihood model.

Selections: *binned* = 5 kpc shells; *complementary* = all stars with centre < r < 305 kpc.

In [ ]:
import csv, os
import numpy as np
import matplotlib.pyplot as plt

DATA_DIR = os.environ.get('DIPOLE_DATA_DIR', '/projectnb/fire-reflex/rpei/data')
TAG = 'm12i-600'

## Style — edit here

In [ ]:
# your palette
PALETTE = {
    'teal_dark':  '#0091ad', 'teal':       '#3fcdda',
    'aqua':       '#83f9f8', 'mint':       '#d6f6eb',
    'cream':      '#fdf1d2', 'sand':       '#f8eaad',
    'pink_light': '#faaaae', 'pink':       '#ff57bb',
}

# which masks to show, their legend label, and colours for (direct, fit)
SERIES = {
    'none':       dict(label='all stars',     direct=PALETTE['pink'],      fit=PALETTE['pink_light']),
    'bound_alex': dict(label='bound removed', direct=PALETTE['teal_dark'], fit=PALETTE['teal']),
}

STYLE_DIRECT = dict(ls='-',  lw=2.0, marker='o', ms=3.5)
STYLE_FIT    = dict(ls='--', lw=1.5)

TITLES = {'binned': 'binned', 'complementary': 'complementary (r > centre, < 305 kpc)'}
SUPTITLE = 'm12i snapshot 600: direct vs fitted $v_{\\rm travel}$'
FIGSIZE = (11, 4.2)
YLIM = (0, None)          # e.g. (0, 120) to clip the all-stars fit spike
SHOW_FIT = True

## Load

In [ ]:
def load(path):
    """{(mask, selection): (centres, |v_travel|)} from a direct or fit CSV."""
    out = {}
    with open(path) as f:
        for r in csv.DictReader(f):
            out.setdefault((r['mask'], r['selection']), []).append(
                (float(r['center_kpc']), float(r['v_travel'])))
    return {k: np.array(sorted(v)).T for k, v in out.items()}

direct = load(os.path.join(DATA_DIR, f'{TAG}-direct.csv'))
fit_path = os.path.join(DATA_DIR, f'{TAG}-fits.csv')
fit = load(fit_path) if os.path.exists(fit_path) else {}
print('direct:', sorted(direct)); print('fit   :', sorted(fit) or 'not found')

## Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=FIGSIZE, sharey=True)
for ax, sel in zip(axes, ['binned', 'complementary']):
    for mask, s in SERIES.items():
        if (mask, sel) in direct:
            c, v = direct[(mask, sel)]
            ax.plot(c, v, color=s['direct'], label=f"{s['label']} (direct)", **STYLE_DIRECT)
        if SHOW_FIT and (mask, sel) in fit:
            c, v = fit[(mask, sel)]
            ax.plot(c, v, color=s['fit'], label=f"{s['label']} (fit)", **STYLE_FIT)
    ax.set_title(TITLES[sel])
    ax.set_xlabel('distance from host centre [kpc]')
    ax.set_ylim(*YLIM)
    ax.grid(alpha=0.25)
axes[0].set_ylabel(r'$|v_{\rm travel}|$ [km/s]')
axes[0].legend(fontsize=8, frameon=False)
fig.suptitle(SUPTITLE, fontsize=12)
fig.tight_layout()
plt.show()

In [ ]:
# save (uncomment)
# fig.savefig(os.path.join(DATA_DIR, f'{TAG}-direct-vs-fit.png'), dpi=200, bbox_inches='tight')